# Multi-gold / openings self-eval

A two-phase loop over **one persistent room** that may hold **0–3 golds** and may or may not have a **boundary opening**. Eating gold does **not** end the session by default: the player commits to a `TARGET:`, walks out an opening when the room is empty, or emits `[END_GAME]` when the room is sealed and empty (or when you explicitly ask it to stop). Sealed one-gold is `Golds: 1` + `Opening: forbid`.

1. **Scenario controls.** Gold-count (`random / 0 / 1 / 2 / 3`) × opening (`require / forbid / random`) applied on **New room**.
2. **Player phase.** Ask / Takeover / Back-to-player after analysis. **Takeover** writes the player reply yourself; it stays on until **Resume agent**.
3. **Analyst phase.** Privileged settings JSON includes an `openings` key. Grades the chosen target, missed forwards, wrong-direction turns, and `[END_GAME]`.
4. **End game** calls `session.force_end("user")` — ends immediately, no analyst grading.

The live row (current frame + maroon scratchpad + teal settings) is user-only. **Edit** / **Render** can be pressed multiple times. Bad JSON stays in edit mode. Generation cannot start while either card is being edited; scene/scratchpad edits are refused while a round is open (finish the round or New room). Edits are hidden from the agent until the next generation.

The **session-state banner** shows `session_state`, golds remaining, openings, and the current `TARGET:` line. When state is anything but `active`, Ask/Analyze grey out until **New room**.

Prereqs: Neo4j up, semantic model seeded, `setup_env.sh`. The first cell connects NAMS and loads Gemma.


In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from agent.multi_gold_session import MultiGoldSelfEvalSession

session = MultiGoldSelfEvalSession(n_gold=None, opening="require", end_on_clear=False)
print("session_id:", session.session_id)
print("session_state:", session.session_state)
if session.logger is not None:
    print("log dir:", session.logger.run_dir)

/venv/main/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/23.9G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

session_id: acc987af7a114f919f089d38fba9b1fe
session_state: active
log dir: logs/self_eval_2026-08-18_17-15-13


In [ ]:
import html as _html
import json
import re

import ipywidgets as widgets
from IPython.display import HTML, Image, clear_output, display

from agent import game_io
from agent.notebook_ui import (
    UiBusy, live_board_row, model_picker, player_takeover_controls,
    room_scenario_bar, tame_shift_enter,
)
from agent.self_eval_session import (
    DEFAULT_ANALYST_QUESTION,
    DEFAULT_PLAYER_QUESTION,
)

_TARGET_RE = re.compile(r"^TARGET:\s*(.+)$", re.MULTILINE)

busy = UiBusy()
room = room_scenario_bar()
gold_dd, opening_dd = room.gold_dd, room.opening_dd
new_room_btn, end_btn = room.new_room_btn, room.end_btn

question_box = widgets.Textarea(
    value=DEFAULT_PLAYER_QUESTION,
    placeholder="Ask the player about the current scene...",
    description="You:",
    layout=widgets.Layout(width="600px", height="70px"),
)
ask_btn = widgets.Button(description="Ask", button_style="primary")
restart_btn = widgets.Button(description="Restart conversation", button_style="warning")
dump_btn = widgets.Button(description="Dump DB status", button_style="info")
out = widgets.Output(layout=widgets.Layout(width="100%"))
banner = widgets.HTML()

analyst_box = widgets.Textarea(
    value=DEFAULT_ANALYST_QUESTION,
    description="Analyst:",
    layout=widgets.Layout(width="600px", height="170px"),
)
analyze_btn = widgets.Button(description="Analyze", button_style="success")
back_btn = widgets.Button(description="Back to player (apply move)", button_style="primary")

FRAME_WIDTH = 420
takeover = None
live = None


def _current_target(raw):
    if not raw:
        return "(none)"
    matches = _TARGET_RE.findall(raw)
    return matches[-1].strip() if matches else "(none)"


def _openings_summary():
    d = game_io.game_to_settings_dict(session.game)
    ops = d.get("openings") or []
    if not ops:
        return "none (sealed)"
    bits = [
        f"{o['side']} w={o['width']:.3f} @ ({o['center'][0]:.2f},{o['center'][1]:.2f})"
        for o in ops
    ]
    return "; ".join(bits)


def _refresh_banner(last_raw=None):
    golds = game_io.gold_remaining(session.game)
    banner.value = (
        "<div style='padding:6px 10px;border:1px solid #888;background:#f7f7f7;"
        "font-family:monospace'>"
        f"<b>state</b>={session.session_state}"
        f" &nbsp; <b>golds</b>={golds}"
        f" &nbsp; <b>openings</b>={_html.escape(_openings_summary())}"
        f" &nbsp; <b>TARGET</b>={_html.escape(_current_target(last_raw))}"
        "</div>"
    )


def _sync_phase():
    active = session.session_state == "active"
    editing = live is not None and live.is_editing()
    gen = busy.generating
    player_on = active and session.phase == "player" and not gen and not editing
    analyst_on = active and session.phase == "analyst" and not gen and not editing
    question_box.disabled = not player_on
    ask_btn.disabled = not player_on or (takeover is not None and takeover.is_on())
    analyst_box.disabled = not analyst_on
    analyze_btn.disabled = not analyst_on
    back_btn.disabled = not (active and session.phase == "analyst" and not gen)
    end_btn.disabled = (not active) or gen
    new_room_btn.disabled = session.phase == "analyst" or gen
    restart_btn.disabled = gen
    if live is not None:
        live.sync_buttons()
    if takeover is not None:
        takeover.sync(player_on)


busy.set_on_change(_sync_phase)
live = live_board_row(session, busy, width=FRAME_WIDTH, on_changed=_sync_phase)


def _show_frame(path, caption):
    if path:
        print(caption)
        display(Image(filename=path, width=FRAME_WIDTH))


def _show_searches(searches):
    for s in searches:
        print(f"  [SEARCH {s['query']}]")


def _show_player_result(result):
    _show_frame(result["before_path"], "-- frame the player saw --")
    _show_searches(result.get("searches", []))
    print(f"Player: {result['raw']}")
    if result["action"]:
        print(f"[pending move: {result['action']} -- NOT applied until the round ends]")
    elif result.get("bare_move"):
        print(
            f"[FORMAT ERROR: bare '{result['bare_move']}' without brackets -- "
            f"NOT a move, nothing will be propagated]"
        )
    else:
        print(
            "[WARNING: no move token detected. "
            "If a move was intended, this is a Format Error]"
        )
    _refresh_banner(result["raw"])
    live.refresh()
    print("\n>>> Analyst phase: edit or submit the question in the lower box.")


def _show_wrong_spans(player_raw, wrong_spans):
    verified = wrong_spans.get("verified", [])
    unverified = wrong_spans.get("unverified", [])
    if verified:
        marked = _html.escape(player_raw)
        for span in verified:
            marked = marked.replace(
                _html.escape(span),
                "<mark>" + _html.escape(span) + "</mark>",
            )
        print("\n-- player reply with VERIFIED wrong spans highlighted --")
        display(HTML(
            "<div style='white-space: pre-wrap; border-left: 3px solid #c00; "
            "padding-left: 8px;'>" + marked + "</div>"
        ))
    if unverified:
        print("\n!!! UNVERIFIED wrong spans (not found verbatim in the player reply):")
        for span in unverified:
            print(f'    WRONG: "{span}"')


def _show_analyst_step(step):
    if step["search_query"] is not None:
        print(f"Analyst: {step['text']}")
        print(f"  [running SEARCH: {step['search_query']}]")
    else:
        print(f"Analyst: {step['text']}")


round_has_analysis = False
_last_player_raw = None


def _ask_player(q, human_reply=None):
    global round_has_analysis, _last_player_raw
    print(f"\n=== You: {q}")
    result = session.ask_player(q, human_reply=human_reply)
    _last_player_raw = result["raw"]
    _show_player_result(result)
    question_box.value = DEFAULT_PLAYER_QUESTION
    analyst_box.value = DEFAULT_ANALYST_QUESTION
    round_has_analysis = False


def on_ask(_):
    q = question_box.value.strip()
    if not q or live.is_editing():
        return
    busy.set_generating(True)
    try:
        with out:
            _ask_player(q)
    finally:
        busy.set_generating(False)


def on_human_submit(raw):
    q = question_box.value.strip() or DEFAULT_PLAYER_QUESTION
    if live.is_editing():
        return
    busy.set_generating(True)
    takeover.sync(False)
    try:
        with out:
            _ask_player(q, human_reply=raw)
    finally:
        busy.set_generating(False)


takeover = player_takeover_controls(on_human_submit, on_mode_change=_sync_phase)


def on_new_room(_):
    global round_has_analysis, _last_player_raw
    if session.phase != "player":
        return
    session.n_gold = gold_dd.value
    session.opening = opening_dd.value
    busy.set_generating(True)
    try:
        with out:
            info = session.reset_game()
            _last_player_raw = None
            print("\n*** New room. ***")
            print(f"n_gold={session.n_gold!r}  opening={session.opening!r}")
            print("openings oracle:")
            print(json.dumps(session.current_settings_dict().get("openings"), indent=2))
            _show_frame(info["frame_path"], "-- board after New room --")
            _refresh_banner()
        live.refresh(force_views=True)
        round_has_analysis = False
    finally:
        busy.set_generating(False)


def on_end(_):
    busy.set_generating(True)
    try:
        with out:
            info = session.force_end("user")
            print("\n*** Session ended by user (no analyst grading). ***")
            _show_frame(info["frame_path"], "-- board at end --")
            _refresh_banner(_last_player_raw)
            print(">>> Press New room to continue.")
        live.refresh()
    finally:
        busy.set_generating(False)


def on_analyze(_):
    global round_has_analysis
    q = analyst_box.value.strip()
    if (not q and round_has_analysis) or live.is_editing():
        return
    busy.set_generating(True)
    try:
        with out:
            print(f"\n=== Analyst question: {q or DEFAULT_ANALYST_QUESTION}")
            result = session.ask_analyst(q, on_step=_show_analyst_step)
            round_has_analysis = True
            _show_wrong_spans(result["player_raw"], result["wrong_spans"])
            print("\n>>> Follow up with the analyst, or press Back to player to apply the move.")
        analyst_box.value = ""
    finally:
        busy.set_generating(False)


def on_back(_):
    busy.set_generating(True)
    try:
        with out:
            result = session.end_round()
            if result["action"] == "END_GAME":
                print("\n[player emitted END_GAME -- session ended]")
                _show_frame(result["frame_path"], "-- board at END_GAME --")
            elif result["action"]:
                print(
                    f"\n[move {result['action']} propagated  "
                    f"gold_collected={result['gold_collected']}  "
                    f"gold_remaining={result['gold_remaining']}]"
                )
                _show_frame(result["frame_path"], "-- board after the propagated move --")
            else:
                print("\n[no pending move to propagate]")
            _refresh_banner(_last_player_raw)
            if session.session_state != "active":
                print(f">>> session_state={session.session_state}; press New room to continue.")
            else:
                print("\n>>> Player phase: ask the next question in the upper box.")
        live.refresh()
    finally:
        busy.set_generating(False)


def on_restart(_):
    global round_has_analysis, _last_player_raw
    session.n_gold = gold_dd.value
    session.opening = opening_dd.value
    info = session.restart()
    with out:
        clear_output()
        print(f"New conversation. session_id={info['session_id']}")
        print("openings oracle:")
        print(json.dumps(session.current_settings_dict().get("openings"), indent=2))
        _show_frame(info["frame_path"], "-- fresh board --")
        _refresh_banner()
    question_box.value = DEFAULT_PLAYER_QUESTION
    analyst_box.value = DEFAULT_ANALYST_QUESTION
    round_has_analysis = False
    _last_player_raw = None
    live.refresh(force_views=True)
    _sync_phase()


def on_dump(_):
    dump_btn.disabled = True
    try:
        with out:
            info = session.dump_db()
            print(f"\nDB dumped -> {info['path']}")
            print(f"  nodes = {info['nodes']}   relationships = {info['relationships']}")
            if session.logger is not None:
                print(f"  llm log: {session.logger.llm_txt}")
                print(f"  db  log: {session.logger.db_txt}")
    finally:
        dump_btn.disabled = False


ask_btn.on_click(on_ask)
new_room_btn.on_click(on_new_room)
end_btn.on_click(on_end)
analyze_btn.on_click(on_analyze)
back_btn.on_click(on_back)
restart_btn.on_click(on_restart)
dump_btn.on_click(on_dump)


def _on_model_switched(info):
    global round_has_analysis, _last_player_raw
    with out:
        if info["restarted"]:
            clear_output()
            print(f"New conversation under {info['label']}. "
                  f"session_id={session.session_id}")
            _show_frame(session.current_frame_path(), "-- fresh board --")
            _refresh_banner()
            live.refresh(force_views=True)
        else:
            print(f"\n[model switched to {info['label']} -- conversation continues]")
    if info["restarted"]:
        question_box.value = DEFAULT_PLAYER_QUESTION
        analyst_box.value = DEFAULT_ANALYST_QUESTION
        round_has_analysis = False
        _last_player_raw = None
    _sync_phase()


display(widgets.VBox([
    model_picker(session, on_switched=_on_model_switched),
    room.box,
    banner,
    live.box,
    out,
    question_box,
    widgets.HBox([ask_btn, takeover.takeover_btn, takeover.resume_btn,
                  restart_btn, dump_btn]),
    takeover.controls_box,
    analyst_box,
    widgets.HBox([analyze_btn, back_btn]),
]))
tame_shift_enter(
    question_box, analyst_box, takeover.reply_box,
    live.scratch.textarea, live.settings.textarea,
)

_sync_phase()
_refresh_banner()
with out:
    print(f"session_id={session.session_id}")
    print("openings oracle:")
    print(json.dumps(session.current_settings_dict().get("openings"), indent=2))
    _show_frame(session.current_frame_path(), "-- starting board --")


### Restore the DB to "semantic seeding only" (optional cleanup)

Episodic wipe, then re-heal `core_player_*` / `core_analyst_*` from the
code seed. Same hatch as `play.ipynb`. Gated by `if False:`.


In [ ]:
if False:
    import shutil

    deleted = session.reset_memory_to_seed()
    print("deleted nodes:", deleted)

    img_dir = session.cfg.image_dir
    if img_dir.exists():
        shutil.rmtree(img_dir, ignore_errors=True)
        print("cleared image dir:", img_dir)

    info = session.restart()
    print("new session_id:", info["session_id"])

When you're done, release the model and close the memory client:

In [ ]:
session.close()